In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from fairlearn.metrics import MetricFrame, selection_rate, true_positive_rate, false_positive_rate

In [ ]:
X = df.drop(columns=[y])
y_arr = df[y].values
A = df['gender']  # protected

num = X.select_dtypes(include=np.number).columns.tolist()
cat = list(set(X.columns) - set(num))
pre = ColumnTransformer([
    ("num","passthrough",num),
    ("cat",OneHotEncoder(handle_unknown="ignore"),cat)
])

clf = Pipeline([("pre", pre), ("lr", LogisticRegression(max_iter=1000))])
X_tr, X_te, y_tr, y_te, A_tr, A_te = train_test_split(X, y_arr, A, test_size=0.2, stratify=y_arr, random_state=7)
clf.fit(X_tr, y_tr)
pred = clf.predict(X_te)

mf = MetricFrame(metrics={
    'selection_rate': selection_rate,
    'tpr': true_positive_rate,
    'fpr': false_positive_rate
}, y_true=y_te, y_pred=pred, sensitive_features=A_te)